In [1]:
# Run this notebooks with chemgifs conda env

In [2]:
import pandas as pd
from tqdm import tqdm
import os
import numpy as np
import tarfile
from collections import Counter

In [3]:
# Define some paths
root = '../../../Documents_GPU/mtb-targeted-protein-degradation/scripts'
PATH_TO_DOCKING_RESULTS_ORIGINAL = os.path.join(root, "..", "processed", "unidock_docking", 'docking_results')
PATH_TO_DOCKING_RESULTS_REAL = os.path.join(root, "..", "processed", "unidock_REAL_docking", 'docking_results')
PATH_TO_INPUT_LIGANDS = os.path.join(root, "..", "processed", "unidock_REAL_docking", 'input_ligands')

# Load pocket detection data
pocket_detection_data = pd.read_csv(os.path.join(root, "..", "processed", "pocket_detection_data.csv"))

In [4]:
DOCKING_RESULTS_ORIGINAL = {}
DOCKING_RESULTS_REAL = {}
DOCKING_RESULTS_REAL_BACKGROUND = {}

# For each pocket
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS_ORIGINAL))):
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS_ORIGINAL, pocket, 'report.csv'), engine='python')
    DOCKING_RESULTS_ORIGINAL[pocket] = {i: j for i, j in zip(scores['compound'], scores['score'])}

# For each pocket
for pocket in tqdm(sorted(os.listdir(PATH_TO_DOCKING_RESULTS_REAL))):
    # try:
    lines = open(os.path.join(PATH_TO_INPUT_LIGANDS, f"input_ligands_{pocket}.txt"), "r").readlines()
    lines = [i.strip().replace(".sdf", "").split("/")[-1] for i in lines]
    actives = set(lines[:100000])
    inactives = set(lines[100000:])
    scores = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'report.csv'))
    scores['set'] = ["inactive" if i in inactives else "active" for i in scores['compound']]
    scores_actives = scores[scores['set'] == 'active'].reset_index(drop=True)
    scores_inactives = scores[scores['set'] == 'inactive'].reset_index(drop=True)
    DOCKING_RESULTS_REAL[pocket] = {i: j for i, j in zip(scores_actives['compound'], scores_actives['score'])}
    DOCKING_RESULTS_REAL_BACKGROUND[pocket] = {i: j for i, j in zip(scores_inactives['compound'], scores_inactives['score'])}
    # except:
    #     pass

100%|██████████| 276/276 [00:31<00:00,  8.66it/s]


In [5]:
### SELECT TOP MOLECULES AND PREPARE GIF USING CHEMGIFS ###

In [6]:
N = 10_000
ACTIVES = []
proteins = set(pocket_detection_data['Uniprot AC'])
ACTIVES_PER_PROTEIN = {i: set() for i in proteins}
for pocket in tqdm(sorted(DOCKING_RESULTS_REAL)):
    act = sorted(DOCKING_RESULTS_REAL[pocket], key = lambda x: DOCKING_RESULTS_REAL[pocket][x])[:N]
    ACTIVES.extend(act)
    ACTIVES_PER_PROTEIN[pocket.split("_")[1]].update(set(act))


# Get multi-target molecules
counts = Counter(ACTIVES)
counts_proteins = Counter([cpd for protein in proteins for cpd in ACTIVES_PER_PROTEIN[protein]])
active_21_proteins = [cmpd for cmpd, c in counts_proteins.items() if c >= 21]
print(f"TOP-{N} actives")
print(f"Compounds that are active at least once: {len(set(ACTIVES))}")
print(f"Compounds that are active in at least 21 proteins: {len(active_21_proteins)}")

# Get ID to SMILES mapping
ID_TO_SMILES = pd.read_csv(os.path.join(root, "..", "processed", "enamine_REAL_characterization", "enamine_REAL.tsv"), sep='\t')
ID_TO_SMILES = {i: j for i,j in zip(ID_TO_SMILES['id'], ID_TO_SMILES['smiles'])}
SMILES = [ID_TO_SMILES[i] for i in active_21_proteins]

  0%|          | 0/276 [00:00<?, ?it/s]

100%|██████████| 276/276 [00:04<00:00, 63.47it/s]


TOP-10000 actives
Compounds that are active at least once: 620557
Compounds that are active in at least 21 proteins: 398


In [18]:
with open(os.path.join(root, "..", "processed", "unidock_REAL_docking", "multi_target_actives_smiles.csv"), "w") as f:
    f.write("smiles\n")
    for smi in SMILES:
        f.write(smi.split()[0] + "\n")

In [7]:
### GET TOP POSES PER POCKET ###

In [ ]:
for pocket in tqdm(sorted(DOCKING_RESULTS_REAL)):

    # Get top molecules
    top_mols = DOCKING_RESULTS_REAL[pocket]
    top_mols = sorted(top_mols, key = lambda x: top_mols[x])[:6]

    # Read tar file and extract top poses
    with tarfile.open(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'docking.tar.gz'), 'r:gz') as tar:

        # Create top poses directory
        os.makedirs(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'top_poses'), exist_ok=True)

        # Get top molecules
        for top_mol in top_mols:
            file = tar.extractfile(f"docking/{top_mol}_out.sdf").read()
            with open(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'top_poses', f"{top_mol}_out.sdf"), "wb") as f:
                f.write(file)

 52%|█████▏    | 144/276 [18:42<17:25,  7.92s/it]

In [9]:
### SELECT TOP MOLECULES FOR VISUALIZATION ###

In [10]:
N = 1_000
ACTIVES = []
proteins = set(pocket_detection_data['Uniprot AC'])
ACTIVES_PER_PROTEIN = {i: set() for i in proteins}
for pocket in tqdm(sorted(DOCKING_RESULTS_REAL)):
    act = sorted(DOCKING_RESULTS_REAL[pocket], key = lambda x: DOCKING_RESULTS_REAL[pocket][x])[:N]
    ACTIVES.extend(act)
    ACTIVES_PER_PROTEIN[pocket.split("_")[1]].update(set(act))


counts = Counter(ACTIVES)
counts_proteins = Counter([cpd for protein in proteins for cpd in ACTIVES_PER_PROTEIN[protein]])
active_10 = [cmpd for cmpd, c in counts.items() if c >= 10]
active_2_proteins = [cmpd for cmpd, c in counts_proteins.items() if c >= 2]
active_10_proteins = [cmpd for cmpd, c in counts_proteins.items() if c >= 10]
active_21_proteins = [cmpd for cmpd, c in counts_proteins.items() if c >= 21]

print(f"TOP-{N} actives")
print(f"Compounds that are active at least once: {len(set(ACTIVES))}")
print(f"Compounds that are active at least 10 times: {len(active_10)}")
print(f"Compounds that are active in at least 2 proteins: {len(active_2_proteins)}")
print(f"Compounds that are active in at least 10 proteins: {len(active_10_proteins)}")
print(f"Compounds that are active in at least 21 proteins: {len(active_21_proteins)}")

100%|██████████| 276/276 [00:04<00:00, 65.57it/s]


TOP-1000 actives
Compounds that are active at least once: 113375
Compounds that are active at least 10 times: 4158
Compounds that are active in at least 2 proteins: 37799
Compounds that are active in at least 10 proteins: 1821
Compounds that are active in at least 21 proteins: 13


In [11]:
for pocket in tqdm(sorted(DOCKING_RESULTS_REAL)):

    # Read tar file and extract top poses
    with tarfile.open(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'docking.tar.gz'), 'r:gz') as tar:

        # Create top poses directory
        os.makedirs(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'multitarget_poses'), exist_ok=True)

        # Get top molecules
        for act in active_21_proteins:
            try:
                file = tar.extractfile(f"docking/{top_mol}_out.sdf").read()
                with open(os.path.join(PATH_TO_DOCKING_RESULTS_REAL, pocket, 'multitarget_poses', f"{act}_out.sdf"), "wb") as f:
                    f.write(file)
            except:
                pass

100%|██████████| 276/276 [25:12<00:00,  5.48s/it]
